# 04 - AWS Cost and Performance Tradeoff Worksheet

This notebook supports the class discussion on **when to use Athena vs EMR** for analytics workloads.

## Goals
- Compare cost and runtime under different query and transformation patterns
- Quantify the impact of partition pruning
- Build a repeatable decision worksheet for project planning

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if Path.cwd().name == 'notebooks':
    REPO_ROOT = Path.cwd().parent
else:
    REPO_ROOT = Path.cwd()

OUT_DIR = REPO_ROOT / 'notebooks' / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Repo root:', REPO_ROOT)
print('Output dir:', OUT_DIR)

## 1) Enter Observed or Assumed Metrics

Edit these values with your own lab or project numbers.

In [ ]:
# Athena pricing (typical): $5 per TB scanned
ATHENA_PRICE_PER_TB = 5.0

# EMR cost assumptions (edit for your cluster)
EMR_CLUSTER_HOURLY_USD = 0.576  # Example: 3 x m4.large (illustrative)
EMR_JOB_MINUTES = 25

# Athena query scan volumes (GB)
ATHENA_SCAN_UNFILTERED_GB = 3.2
ATHENA_SCAN_FILTERED_GB = 0.15

# Runtime assumptions (seconds)
ATHENA_RUNTIME_UNFILTERED_SEC = 16
ATHENA_RUNTIME_FILTERED_SEC = 3
EMR_RUNTIME_SEC = EMR_JOB_MINUTES * 60

# Query frequency assumptions
QUERIES_PER_DAY_UNFILTERED = 20
QUERIES_PER_DAY_FILTERED = 120

In [ ]:
def athena_query_cost_usd(scan_gb: float, price_per_tb: float = ATHENA_PRICE_PER_TB) -> float:
    return (scan_gb / 1024.0) * price_per_tb


def emr_job_cost_usd(hourly_rate: float = EMR_CLUSTER_HOURLY_USD, minutes: float = EMR_JOB_MINUTES) -> float:
    return hourly_rate * (minutes / 60.0)


athena_unfiltered_cost = athena_query_cost_usd(ATHENA_SCAN_UNFILTERED_GB)
athena_filtered_cost = athena_query_cost_usd(ATHENA_SCAN_FILTERED_GB)
emr_cost = emr_job_cost_usd()

summary = pd.DataFrame([
    {
        'mode': 'Athena (unfiltered query)',
        'cost_usd': round(athena_unfiltered_cost, 4),
        'runtime_sec': ATHENA_RUNTIME_UNFILTERED_SEC,
    },
    {
        'mode': 'Athena (partition-filtered query)',
        'cost_usd': round(athena_filtered_cost, 4),
        'runtime_sec': ATHENA_RUNTIME_FILTERED_SEC,
    },
    {
        'mode': 'EMR batch job',
        'cost_usd': round(emr_cost, 4),
        'runtime_sec': EMR_RUNTIME_SEC,
    },
])

summary

## 2) Daily Cost Projection

This compares frequent query workloads vs scheduled batch processing.

In [ ]:
daily_cost_athena_unfiltered = athena_unfiltered_cost * QUERIES_PER_DAY_UNFILTERED
daily_cost_athena_filtered = athena_filtered_cost * QUERIES_PER_DAY_FILTERED
daily_cost_emr_batch = emr_cost  # assuming one batch run/day

projection = pd.DataFrame([
    {'workload': 'Athena unfiltered (daily)', 'daily_cost_usd': daily_cost_athena_unfiltered},
    {'workload': 'Athena filtered (daily)', 'daily_cost_usd': daily_cost_athena_filtered},
    {'workload': 'EMR batch (daily)', 'daily_cost_usd': daily_cost_emr_batch},
]).sort_values('daily_cost_usd', ascending=False)

projection['daily_cost_usd'] = projection['daily_cost_usd'].round(4)
projection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

summary.plot.bar(x='mode', y='cost_usd', ax=axes[0], legend=False, color=['#9C27B0', '#4CAF50', '#FB8C00'])
axes[0].set_title('Per-Operation Cost')
axes[0].set_ylabel('USD')
axes[0].tick_params(axis='x', labelrotation=20)

summary.plot.bar(x='mode', y='runtime_sec', ax=axes[1], legend=False, color=['#9C27B0', '#4CAF50', '#FB8C00'])
axes[1].set_title('Runtime Comparison')
axes[1].set_ylabel('Seconds')
axes[1].tick_params(axis='x', labelrotation=20)

plt.tight_layout()
plt.show()

## 3) Break-Even Query Frequency

How many Athena queries at your average scan size equal one EMR batch run cost?

In [ ]:
AVG_SCAN_GB_PER_ATHENA_QUERY = 0.25  # adjust to your real query profile
per_query_cost = athena_query_cost_usd(AVG_SCAN_GB_PER_ATHENA_QUERY)

if per_query_cost > 0:
    break_even_queries = emr_cost / per_query_cost
else:
    break_even_queries = np.inf

print(f'Average Athena query scan: {AVG_SCAN_GB_PER_ATHENA_QUERY:.3f} GB')
print(f'Athena cost/query: ${per_query_cost:.5f}')
print(f'EMR batch run cost: ${emr_cost:.4f}')
print(f'Break-even query count vs one EMR run: {break_even_queries:.1f} queries')

## 4) Decision Worksheet

Use this template to justify platform choice by workload.

In [ ]:
decision_template = pd.DataFrame([
    {'workload_type': 'Ad-hoc analyst SQL', 'recommended': 'Athena', 'reason': 'Serverless, low ops, fast iteration'},
    {'workload_type': 'Large daily ETL/feature build', 'recommended': 'EMR', 'reason': 'Distributed transforms and controlled batch windows'},
    {'workload_type': 'Interactive dashboard on curated tables', 'recommended': 'Athena', 'reason': 'On-demand query serving with partition pruning'},
    {'workload_type': 'Heavy joins + custom Spark logic', 'recommended': 'EMR', 'reason': 'Execution control and Spark-native optimizations'},
])

decision_template

In [ ]:
summary_out = OUT_DIR / 'cost_tradeoff_summary.csv'
projection_out = OUT_DIR / 'cost_tradeoff_daily_projection.csv'

decision_out = OUT_DIR / 'cost_tradeoff_decision_template.csv'

summary.to_csv(summary_out, index=False)
projection.to_csv(projection_out, index=False)
decision_template.to_csv(decision_out, index=False)

print('Wrote:')
print('-', summary_out)
print('-', projection_out)
print('-', decision_out)

## Reflection Prompts
1. Which variable drives your cost the most: scan size, query count, or cluster runtime?
2. What storage/layout changes (for example partitioning) reduce Athena spend most effectively?
3. When does EMR become justified even if it costs more per run?
4. How would this analysis change for BigQuery + Dataproc in Google Cloud?